# Delivery Optimization

## Objective
- Model and solve a realistic Delivery Optiomization Problem with Time Windows (VRPTW) for urban last-mile delivery optimization by leveraging real-world data from multiple Kaggle datasets.
- Compare solution quality, runtime, scalability, and feasibility of each exact method and heuristic method.

In [ ]:
# ===== IMPORTS =====
from __future__ import annotations
import argparse
import math
import time
from dataclasses import dataclass
from typing import List, Tuple
import numpy as np
import pandas as pd
import pulp
import warnings
warnings.filterwarnings("ignore")

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


## Data Loading and Preprocessing

In [ ]:
def load_and_prepare_data(csv_path: str, n_orders: int, seed: int) -> Tuple[Tuple[float, float], pd.DataFrame]:
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    for col in ["Traffic", "Weather", "Vehicle", "Area"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

    required_cols = [
        "Store_Latitude",
        "Store_Longitude",
        "Drop_Latitude",
        "Drop_Longitude",
        "Traffic",
        "Weather",
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    work = df.dropna(subset=required_cols).copy()
    work = work[~work["Traffic"].str.lower().eq("nan")]
    work = work[~work["Weather"].str.lower().eq("nan")]

    store_group = (
        work.groupby(["Store_Latitude", "Store_Longitude"]).size().reset_index(name="count")
    )
    busiest = store_group.sort_values("count", ascending=False).iloc[0]
    depot = (float(busiest["Store_Latitude"]), float(busiest["Store_Longitude"]))

    subset = work[
        (work["Store_Latitude"] == depot[0]) & (work["Store_Longitude"] == depot[1])
    ].copy()

    if len(subset) < n_orders:
        n_orders = len(subset)

    subset = subset.sample(n=n_orders, random_state=seed).reset_index(drop=True)
    return depot, subset

## Kavishka Code

## Arosha Code

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2.  COST MATRIX
# ─────────────────────────────────────────────────────────────

def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Great-circle distance in kilometres (Haversine formula)."""
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


def build_cost_matrix(
    depot: Tuple[float, float], orders: pd.DataFrame
) -> np.ndarray:
    """
    Build an (n+1)×(n+1) weighted cost matrix.
    Node 0 = depot; nodes 1..n = delivery locations.
    cost[i][j] = haversine(i→j) × traffic_mult[j] × weather_mult[j]
    Destination conditions (traffic, weather) affect traversal cost.
    """
    lats  = [depot[0]] + orders["Drop_Latitude"].tolist()
    lons  = [depot[1]] + orders["Drop_Longitude"].tolist()
    mults = [1.0]      + (orders["traffic_mult"] * orders["weather_mult"]).tolist()

    n = len(lats)
    C = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(n):
            if i != j:
                C[i][j] = haversine_km(lats[i], lons[i], lats[j], lons[j]) * mults[j]
    return C


def route_cost(route: List[int], C: np.ndarray) -> float:
    """Total weighted cost of the circuit: depot → route → depot."""
    if not route:
        return float("inf")
    total  = C[0][route[0]]
    total += sum(C[route[k]][route[k + 1]] for k in range(len(route) - 1))
    total += C[route[-1]][0]
    return total


# ─────────────────────────────────────────────────────────────
# 3.  SOLVER 1 – HELD-KARP DYNAMIC PROGRAMMING (Exact)
# ─────────────────────────────────────────────────────────────

def solve_held_karp(C: np.ndarray) -> dict:
    """
    Held-Karp exact TSP algorithm using bitmask DP.

    Formulation
    -----------
    State  : dp[S][i] = minimum cost to reach node i having visited
             exactly the nodes represented by bitmask S, starting from depot.
             Depot = node 0 (implicit); deliveries = nodes 1..n (bits 0..n-1).
    Base   : dp[1<<i][i] = C[0][i+1]   for each delivery i in 0..n-1
    Trans  : dp[S|1<<j][j] = min(dp[S|1<<j][j],
                                  dp[S][i] + C[i+1][j+1])
             for all i in S, j not in S
    Answer : min over i of  dp[FULL][i] + C[i+1][0]

    Complexity
    ----------
    Time  : O(n² · 2ⁿ)  –  exact and globally optimal
    Space : O(n · 2ⁿ)   –  bitmask DP table

    Practical limit: n ≤ 20 comfortably; n = 22-23 within minutes.
    Use --skip-dp and rely on GA for larger instances.
    """
    n_total = C.shape[0]
    n = n_total - 1             # number of delivery nodes (excluding depot)

    if n == 0:
        return {
            "solver": "Held-Karp DP (Exact)", "route": [], "cost": 0.0,
            "runtime_s": 0.0, "feasible": True, "status": "Trivial (0 nodes)",
        }

    INF  = float("inf")
    FULL = (1 << n) - 1        # bitmask with all n delivery bits set

    # dp[mask][i] : min cost to visit nodes in `mask`, ending at delivery i
    # parent[mask][i] : predecessor index for traceback
    dp     = np.full((1 << n, n), INF,  dtype=np.float64)
    parent = np.full((1 << n, n), -1,   dtype=np.int32)

    t0 = time.perf_counter()

    # ── base: depot → single delivery node ───────────────────────
    for i in range(n):
        dp[1 << i][i] = C[0, i + 1]

    # ── DP transition ─────────────────────────────────────────────
    for mask in range(1, 1 << n):
        for i in range(n):
            if not (mask & (1 << i)):
                continue
            cur_cost = dp[mask][i]
            if cur_cost == INF:
                continue
            for j in range(n):
                if mask & (1 << j):      # already visited
                    continue
                new_mask = mask | (1 << j)
                new_cost = cur_cost + C[i + 1, j + 1]
                if new_cost < dp[new_mask][j]:
                    dp[new_mask][j]     = new_cost
                    parent[new_mask][j] = i

    # ── find cheapest return to depot ────────────────────────────
    best_cost  = INF
    last_node  = -1
    for i in range(n):
        if dp[FULL][i] == INF:
            continue
        total = dp[FULL][i] + C[i + 1, 0]
        if total < best_cost:
            best_cost = total
            last_node = i

    # ── traceback to reconstruct route ───────────────────────────
    route_idx = []
    mask, node = FULL, last_node
    while node != -1:
        route_idx.append(node)
        prev = parent[mask][node]
        mask = mask ^ (1 << node)
        node = prev
    route_idx.reverse()

    # delivery indices → node numbers (1..n)
    route   = [idx + 1 for idx in route_idx]
    elapsed = time.perf_counter() - t0

    return {
        "solver":     "Held-Karp DP (Exact)",
        "route":      route,
        "cost":       best_cost,
        "runtime_s":  elapsed,
        "feasible":   len(route) == n,
        "status":     "Globally optimal",
    }


# ─────────────────────────────────────────────────────────────
# 4.  SOLVER 2 – GENETIC ALGORITHM (Heuristic)
# ─────────────────────────────────────────────────────────────

class GeneticTSP:
    """
    Permutation-based Genetic Algorithm for TSP.

    Representation  : chromosome = permutation of delivery nodes (1..n).
                      Depot (node 0) is implicit at start/end.

    Operators
    ---------
    Initialisation : random permutations + one greedy nearest-neighbour seed.
    Selection      : tournament selection (size k) – preserves diversity.
    Crossover      : Order Crossover OX1 – copies a slice from parent-1,
                     fills the rest in parent-2's order; always valid permutation.
    Mutation       : swap mutation – O(1), swaps two random positions.
    Local search   : 2-opt on elite chromosomes every `two_opt_freq` generations;
                     exploits promising regions efficiently.
    Elitism        : top `elite_frac` fraction carried unchanged each generation.

    Complexity  : O(pop_size × n_gen × n)
    Scalability : practical for hundreds–thousands of nodes.
    """

    def __init__(
        self,
        C: np.ndarray,
        pop_size: int     = 150,
        n_gen: int        = 800,
        cx_prob: float    = 0.85,
        mut_prob: float   = 0.15,
        tournament_k: int = 5,
        elite_frac: float = 0.10,
        two_opt_freq: int = 50,
        seed: int         = 42,
        time_limit: float = 60.0,
    ):
        self.C           = C
        self.n           = C.shape[0] - 1
        self.pop_size    = pop_size
        self.n_gen       = n_gen
        self.cx_prob     = cx_prob
        self.mut_prob    = mut_prob
        self.k           = tournament_k
        self.n_elite     = max(2, int(elite_frac * pop_size))
        self.two_opt_freq = two_opt_freq
        self.rng         = np.random.default_rng(seed)
        self.time_limit  = time_limit

    # ── vectorised fitness ────────────────────────────────────────
    def _fitness(self, chrom: np.ndarray) -> float:
        return (
            self.C[0, chrom[0]]
            + float(np.sum(self.C[chrom[:-1], chrom[1:]]))
            + self.C[chrom[-1], 0]
        )

    # ── greedy nearest-neighbour seed ────────────────────────────
    def _greedy_tour(self) -> np.ndarray:
        unvisited = list(range(1, self.n + 1))
        tour, cur = [], 0
        while unvisited:
            nxt = min(unvisited, key=lambda j: self.C[cur, j])
            tour.append(nxt); unvisited.remove(nxt); cur = nxt
        return np.array(tour)

    # ── initial population ────────────────────────────────────────
    def _init_population(self) -> List[np.ndarray]:
        nodes = np.arange(1, self.n + 1)
        pop   = [self.rng.permutation(nodes) for _ in range(self.pop_size)]
        pop[0] = self._greedy_tour()   # seed one good individual
        return pop

    # ── tournament selection ──────────────────────────────────────
    def _tournament(self, pop: List[np.ndarray], fits: np.ndarray) -> np.ndarray:
        idx  = self.rng.choice(len(pop), self.k, replace=False)
        best = idx[int(np.argmin(fits[idx]))]
        return pop[best].copy()

    # ── OX1 crossover ────────────────────────────────────────────
    def _ox1(self, p1: np.ndarray, p2: np.ndarray) -> np.ndarray:
        n     = len(p1)
        a, b  = sorted(self.rng.choice(n, 2, replace=False))
        child = np.full(n, -1, dtype=int)
        child[a:b] = p1[a:b]
        fill  = [g for g in p2 if g not in child]
        ptr   = 0
        for i in range(n):
            if child[i] == -1:
                child[i] = fill[ptr]; ptr += 1
        return child

    # ── swap mutation ─────────────────────────────────────────────
    def _swap_mutate(self, chrom: np.ndarray) -> np.ndarray:
        i, j       = self.rng.choice(len(chrom), 2, replace=False)
        chrom[i], chrom[j] = chrom[j], chrom[i]
        return chrom

    # ── 2-opt local search ────────────────────────────────────────
    def _two_opt(self, chrom: np.ndarray) -> np.ndarray:
        best, best_cost = chrom.copy(), self._fitness(chrom)
        improved = True
        while improved:
            improved = False
            for i in range(len(best) - 1):
                for j in range(i + 2, len(best)):
                    cand         = best.copy()
                    cand[i:j+1]  = best[i:j+1][::-1]
                    c = self._fitness(cand)
                    if c < best_cost - 1e-9:
                        best, best_cost = cand, c
                        improved = True
        return best

    # ── main loop ─────────────────────────────────────────────────
    def run(self) -> dict:
        t0   = time.perf_counter()
        pop  = self._init_population()
        fits = np.array([self._fitness(c) for c in pop])

        best_cost  = float(np.min(fits))
        best_chrom = pop[int(np.argmin(fits))].copy()
        history    = [best_cost]

        print(f"  GA gen {'0':>4} | best cost = {best_cost:.4f} km  "
              f"(pop={self.pop_size}, seeded with greedy tour)")

        for gen in range(1, self.n_gen + 1):
            if time.perf_counter() - t0 > self.time_limit:
                print(f"  GA stopped at gen {gen} – time limit reached.")
                break

            # elitism
            elite_idx = np.argsort(fits)[: self.n_elite]
            new_pop   = [pop[i].copy() for i in elite_idx]

            # 2-opt on best elite every two_opt_freq generations
            if gen % self.two_opt_freq == 0:
                new_pop[0]          = self._two_opt(new_pop[0])
                fits[elite_idx[0]]  = self._fitness(new_pop[0])

            # crossover + mutation to fill rest of population
            while len(new_pop) < self.pop_size:
                p1    = self._tournament(pop, fits)
                p2    = self._tournament(pop, fits)
                child = self._ox1(p1, p2) if self.rng.random() < self.cx_prob else p1.copy()
                if self.rng.random() < self.mut_prob:
                    child = self._swap_mutate(child)
                new_pop.append(child)

            pop  = new_pop
            fits = np.array([self._fitness(c) for c in pop])

            gen_best = float(np.min(fits))
            if gen_best < best_cost:
                best_cost  = gen_best
                best_chrom = pop[int(np.argmin(fits))].copy()
            history.append(best_cost)

            if gen % 100 == 0:
                print(f"  GA gen {gen:>4} | best cost = {best_cost:.4f} km")

        elapsed = time.perf_counter() - t0
        return {
            "solver":      "Genetic Algorithm (OX1 + 2-opt)",
            "route":       best_chrom.tolist(),
            "cost":        best_cost,
            "runtime_s":   elapsed,
            "feasible":    True,
            "status":      "Near-optimal heuristic",
            "convergence": history,
        }


# ─────────────────────────────────────────────────────────────
# 5.  REPORTING
# ─────────────────────────────────────────────────────────────

def print_report(results: list, orders: pd.DataFrame) -> None:
    print("\n" + "=" * 72)
    print("DELIVERY OPTIMIZATION – RESULTS COMPARISON")
    print("=" * 72)
    print(f"{'Solver':<38} {'Cost(km)':>10} {'Runtime(s)':>11} {'Feasible':>9}  Status")
    print("-" * 72)
    for r in results:
        cost_s = f"{r['cost']:.4f}" if r["cost"] < 1e15 else "N/A"
        print(
            f"{r['solver']:<38} {cost_s:>10} {r['runtime_s']:>11.4f}"
            f" {'Yes' if r['feasible'] else 'No':>9}  {r['status']}"
        )
    print("=" * 72)

    # route details
    for r in results:
        if not r["feasible"] or not r["route"]:
            continue
        print(f"\n── {r['solver']} Route ──")
        stops = []
        for node in r["route"]:
            row  = orders.loc[node - 1]
            area = str(row.get("Area", "")).strip()
            stops.append(
                f"[{node}:{row['Drop_Latitude']:.3f},{row['Drop_Longitude']:.3f} {area}]"
            )
        print("  Depot → " + " → ".join(stops) + " → Depot")
        print(f"  Total weighted cost : {r['cost']:.4f} km")

    # optimality gap
    dp_res = [r for r in results if "Held-Karp" in r["solver"] and r["feasible"]]
    if dp_res:
        dp_cost = dp_res[0]["cost"]
        print("\n── Optimality Gap vs Held-Karp DP ──")
        for r in results:
            if "Held-Karp" not in r["solver"] and r["feasible"]:
                gap = (r["cost"] - dp_cost) / dp_cost * 100
                print(f"  {r['solver']:<38}: {gap:+.4f}%")

    # scalability
    print("\n── Scalability & Complexity ──")
    print("  Held-Karp DP : O(n² · 2ⁿ) time, O(n · 2ⁿ) space.")
    print("                 Globally optimal. Practical limit: n ≤ 20.")
    print("  Genetic Algo : O(pop × gens × n) time. Near-optimal.")
    print("                 Scales to hundreds–thousands of nodes.")

    feasible = [r for r in results if r["feasible"]]
    if feasible:
        best = min(feasible, key=lambda r: r["cost"])
        print(f"\n✓  Best solution : {best['solver']}  |  Cost : {best['cost']:.4f} km (weighted)")


# ─────────────────────────────────────────────────────────────
# 6.  MAIN
# ─────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="Delivery Route Optimization")
    parser.add_argument("--data",       default="amazon_delivery.csv",
                        help="Path to Amazon delivery CSV")
    parser.add_argument("--orders",     type=int, default=14,
                        help="Number of orders to optimise "
                             "(Held-Karp is exact but exponential – keep ≤ 20)")
    parser.add_argument("--seed",       type=int, default=42,
                        help="Random seed for reproducibility")
    parser.add_argument("--time-limit", type=int, default=120,
                        help="Time limit for the GA in seconds")
    parser.add_argument("--skip-dp",    action="store_true",
                        help="Skip Held-Karp DP (use for n > 20)")
    args = parser.parse_args()

    # load & preprocess
    depot, orders = load_and_prepare_data(args.data, args.orders, args.seed)

    # build cost matrix
    print("[Building cost matrix…]")
    C = build_cost_matrix(depot, orders)
    n = len(orders)
    print(f"[Cost matrix] {n+1}×{n+1}  (depot + {n} deliveries)\n")

    results = []

    # ── SOLVER 1: Held-Karp DP ────────────────────────────────────
    if not args.skip_dp:
        if n > 20:
            print(f"[1/2] WARNING: n={n} > 20; Held-Karp O(n²·2ⁿ) may be very slow. "
                  "Pass --skip-dp to use GA only.")
        print("[1/2] Running Held-Karp Dynamic Programming (exact)…")
        dp_res = solve_held_karp(C)
        print(f"      Done.  Cost={dp_res['cost']:.4f} km  "
              f"Time={dp_res['runtime_s']:.4f}s  Status={dp_res['status']}")
        results.append(dp_res)
    else:
        print("[1/2] Held-Karp DP skipped.")

    # ── SOLVER 2: Genetic Algorithm ──────────────────────────────
    print(f"\n[2/2] Running Genetic Algorithm (time limit={args.time_limit}s)…")
    ga = GeneticTSP(
        C,
        pop_size      = 150,
        n_gen         = 800,
        cx_prob       = 0.85,
        mut_prob      = 0.15,
        tournament_k  = 5,
        elite_frac    = 0.10,
        two_opt_freq  = 50,
        seed          = args.seed,
        time_limit    = float(args.time_limit),
    )
    ga_res = ga.run()
    print(f"\n      Done.  Cost={ga_res['cost']:.4f} km  "
          f"Time={ga_res['runtime_s']:.4f}s  Status={ga_res['status']}")
    results.append(ga_res)

    # final report
    print_report(results, orders)


if __name__ == "__main__":
    main()